<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Machine-Learning/19-interpretability-robustness-fairness-privacy.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回机器学习总览](Machine Learning.html)


## **可解释性、鲁棒性、公平性与隐私**

一个模型即使在基准测试上很准确，也可能并不适合实际使用。它可能依赖泄漏特征、在环境变化时失效、把错误集中到某个较小群体、记忆训练记录，或者产生任何负责人都无法合理申诉与复核的决策。这些并不是“真正建模工作”结束后才附加的罕见边界情况；它们决定了测得的性能在模型真正采取行动的场景中是否值得信任。

本章把四类要求区分开来：

- **可解释性（interpretability）**询问模型学到了怎样的行为，以及解释对于预期受众和决策目的是否足够忠实。
- **鲁棒性（robustness）**询问在扰动、分布偏移、缺失、对抗行为与运行故障下，模型性能会怎样变化。
- **公平性（fairness）**询问收益、负担、错误和决策规则如何分配给不同的人，尤其是面临特定伤害的群体。
- **隐私（privacy）**询问能否从数据、参数、梯度和输出中推断出有关个人、记录、客户端或总体的信息。

<div class="diagram-scroll">

![可解释性、鲁棒性、公平性与隐私是彼此独立的可靠性维度。](assets/trustworthy-ml-dimensions.svg){fig-alt="四个方框分别概括可解释性、鲁棒性、公平性与隐私，每个维度对应不同的评估问题。"}

</div>

通过一个维度并不能认证另一个维度。透明的线性模型可能存在歧视；满足差分隐私的模型可能对某个群体不准确；鲁棒分类器仍可能泄露成员身份；汇总公平指标合格也可能掩盖任意的局部行为。因此，有效审查应从一张**主张—证据映射（claim-evidence map）**开始：

| 主张 | 相关证据 | 单独使用时不充分的证据 |
|---|---|---|
| “模型可以理解” | 明确的受众、解释对象、保真度检查、稳定性检查与已知限制 | 一张颜色鲜艳的特征重要性图 |
| “模型具有鲁棒性” | 明确的偏移或攻击者、严重程度范围、切片级退化、不确定性与回退行为 | 干净测试集准确率 |
| “模型是公平的” | 伤害模型、受影响群体、经过论证的指标、不确定性与缓解措施的影响 | 看过结果后才挑选的一个平等指标 |
| “模型保护隐私” | 隐私单元、相邻关系、威胁模型、发布接口、隐私核算与实现审计 | 数据仍留在本地设备上 |

模型只是一个**社会技术系统（sociotechnical system）**中的组件。数据收集、标签、用户行为、阈值、人工审核、访问控制、申诉与反馈循环都可能主导系统结果。因此，评估必须覆盖完整决策过程，而不能停在拟合好的估计器上。

### **可解释性与解释**

*Interpretability* 与 *explainability* 的使用并不统一，因此应直接说明想提出什么主张。本章采用以下含义：

- **可解释性**是人能够理解模型相关行为的程度，例如变量怎样影响预测，或是哪条规则产生了决策。
- **解释（explanation）**是针对某个预测、模型行为或受众生成的产物。
- **透明度（transparency）**涉及对数据、模型设计、训练、所有权和决策政策的访问与清晰说明。
- **忠实性（faithfulness）**衡量解释是否准确反映被解释模型的真实行为。
- **合理性（plausibility）**衡量解释在人看来是否听起来合理。

忠实性并不等于合理性。一个听起来合理的解释可能使用模型从未采用的概念来为预测找理由；反过来，忠实解释可能暴露出不理想且难以沟通的依赖关系。因此，必须相对于精确定义的对象检验解释质量：

1. **对象：**预测、分数、排序、表示、训练样本或政策。
2. **受众：**开发者、领域专家、审计人员、操作人员或受影响者。
3. **用途：**调试、科学理解、验证、补救、合规或监控。
4. **保真区域：**一个点、局部邻域、子群体或整个输入分布。
5. **输出尺度：**概率、对数几率、原始分数、类别或效用。

一个解释可能对其中一种契约正确，却对另一种契约产生误导。例如，局部线性近似可以解释某位申请人在邻域内的原始分数，但不能说明全局单调性，也不能说明改变申请人属性的因果效应。

#### **内在解释与事后解释**

**内在可解释模型（intrinsically interpretable model）**直接暴露其决策结构，例如稀疏线性模型、短决策列表、浅层树、广义加性模型或单调评分卡。可解释性并不是算法名称自带的属性。一个包含 50,000 个相关特征与不透明预处理的线性模型并不真正便于检查；一个受到精心约束的非线性模型反而可能更清楚。

**事后解释（post-hoc explanation）**在训练完成后分析拟合模型。它可以扰动输入、拟合局部代理模型、分解预测、检索有影响力的训练样本、可视化激活，或搜索反事实。面对复杂预测器时，事后方法很有价值，但它会引入第二个估计问题：解释本身也有假设、近似误差、随机性和参考分布。

核心验证问题是：

> 如果解释声称某个特征或规则很重要，那么改变或移除该部分后，模型在声明区域内是否会按照解释预测的方式变化？

可用检查包括扰动测试、代理模型保真度、不同随机种子的重复运行、自助法不确定性、对背景数据集的敏感性，以及与已知合成真值的比较。两个解释工具得出一致结论是积极信号，但不是证明，因为二者可能共享同一个无效的独立性假设。

#### **全局解释与局部解释**

**全局解释（global explanation）**概括模型在一个分布上的行为，例如整体特征重要性、响应曲线、紧凑的代理树、规则集合或子群体行为。**局部解释（local explanation）**描述一个预测或较小邻域，例如特征贡献、局部代理、反事实或有影响力的样本。

<div class="diagram-scroll">

![全局与局部、内在与事后是彼此独立的解释维度。](assets/explanation-scope-map.svg){fig-alt="一个二乘二概念图分别区分解释的全局与局部范围，以及内在与事后的解释机制。"}

</div>

数据分布本身就是全局解释的一部分。“特征 $j$ 很重要”实际表示：在某个模型、指标、扰动方式和总体下，它很重要。部署群体构成变化后，解释也可能变化。局部解释同样需要定义邻域；对于文本、图像、编码后的类别或相关临床指标，原始输入空间中的距离可能毫无意义。

不能把模型解释提升为因果结论。大多数特征归因方法解释的是 $f(x)$，而不是 $Y(do(X_j=x_j'))$。如果收入与教育相关，在保持另一个变量不变时替换其中之一，可能构造出不合理的个体；即使模型输入的改变在统计上合理，也不能证明现实干预会导致怎样的结果。

**对比。**内在模型提供直接的结构证据，但可能需要限制容量。事后方法支持复杂模型，却需要保真度检验。全局解释支持验证与治理；局部解释支持调试和个体复核。负责任的分析往往同时需要两者，因为全局可接受的模型仍可能包含局部伤害，而局部听起来合理的解释也可能掩盖糟糕的全局政策。


### **模型特定解释**

模型特定方法利用估计器自身的数学结构。它们通常比黑盒扰动方法更快、更精确，但其含义仍然取决于特征构造、输出尺度、正则化与变量之间的依赖。

#### **线性系数与树结构**

对于线性回归，

$$
\hat y=\beta_0+\sum_{j=1}^{p}\beta_jx_j,
$$

$\beta_j$ 表示：在其他已编码特征保持不变时，$x_j$ 增加一个单位所对应的拟合输出变化。这句话包含多个限制：

- 除非采用有意义的标准化，否则不同单位特征的系数不能直接比较；
- one-hot 系数是相对于被省略的参考类别而言的；
- 存在交互时，某个特征的效应取决于其他变量；
- 强共线性可能让单个系数不稳定，即使预测本身很稳定；
- 正则化通过同时优化预测损失与惩罚项来改变系数；
- “保持其他变量不变”描述的是关联关系，而且可能对应不可能出现的组合。

对于逻辑回归，

$$
\log\frac{P(Y=1\mid x)}{1-P(Y=1\mid x)}
=\beta_0+\sum_j\beta_jx_j.
$$

因此，$\exp(\beta_j)$ 是增加一个单位时的**条件几率比（conditional odds ratio）**，而不是概率差。固定的对数几率变化在概率接近 0 或 1 时影响较小，在概率接近 0.5 时影响较大：

$$
\frac{\partial P(Y=1\mid x)}{\partial x_j}
=\beta_j\,p(x)\bigl(1-p(x)\bigr).
$$

解释决策树时，需要沿根节点到叶节点的精确路径前进。每个分裂都会缩小特征空间区域，叶节点预测则概括该区域中的训练观测。可以通过深度、叶节点数、分裂阈值、每个叶节点的样本支持，以及特征被重复使用的情况来检查全局结构。不过，基于不纯度的重要性偏好具有更多候选切分点的特征，也可能在可替代特征之间以不稳定方式分配贡献。

<details>
<summary><strong>Python：审计线性系数与精确的树决策路径</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

rng = np.random.default_rng(19)
X, y = make_classification(
    n_samples=1600,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    class_sep=1.1,
    random_state=19,
)

# Put feature 0 on a much larger numerical scale. Raw coefficient magnitude
# will shrink even though the information carried by the feature is unchanged.
X_scaled_units = X.copy()
X_scaled_units[:, 0] *= 100.0

raw_model = LogisticRegression(max_iter=2000).fit(X_scaled_units, y)
standardizer = StandardScaler().fit(X_scaled_units)
standard_model = LogisticRegression(max_iter=2000).fit(
    standardizer.transform(X_scaled_units), y
)

print("Raw coefficients:", np.round(raw_model.coef_[0], 4))
print("Standardized coefficients:", np.round(standard_model.coef_[0], 4))
print("Standardized odds ratios:", np.round(np.exp(standard_model.coef_[0]), 3))

# A shallow tree exposes the exact rule used for a selected prediction.
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=60, random_state=19)
tree.fit(X, y)
sample = X[[0]]
leaf_id = tree.apply(sample)[0]
path_nodes = tree.decision_path(sample).indices

print("\nTree rules:\n", export_text(tree, feature_names=[f"x{i}" for i in range(4)]))
print("Selected sample:", np.round(sample[0], 3))
print("Visited node IDs:", path_nodes.tolist())
print("Leaf ID and predicted probability:", leaf_id, np.round(tree.predict_proba(sample)[0], 3))
```

</details>

标准化系数比较的是训练分布下移动一个标准差的结果；它不会让特征自动具有因果性或可行动性。树路径对这棵已拟合的树是精确的，但轻微数据扰动也可能改变树结构。应在不同折或 bootstrap 样本上重新拟合，比较所选特征、阈值和预测，从而评估稳定性。

有意识地设置约束时，模型特定解释最有价值：

| 模型设计 | 可解释性收益 | 仍然存在的风险 |
|---|---|---|
| 稀疏线性或逻辑回归 | 只有少量加性项 | 相关性与非线性误设 |
| 广义加性模型 | 平滑的单特征函数 | 必须显式加入交互 |
| 单调模型 | 强制方向关系 | 方向可能错误或不完整 |
| 浅层树或决策列表 | 人类可阅读的规则 | 不稳定与粗糙边界 |
| 有界分值的评分卡 | 运行透明度 | 离散化与阈值效应 |

**总结。**应在正确尺度上阅读系数，检查整个预处理流水线，并区分“对已拟合结构的精确解释”和“在可能训练样本之间的稳定性”。当简单模型的约束仍能保留足够效用，并让关键行为可审计时，应优先选择它；而且复杂度应在真正呈现给人的表示空间中衡量。


### **模型无关解释**

模型无关方法把拟合好的预测器视为一个可以查询的函数。这样做使方法能够广泛复用，但“无关”并不等于“没有假设”。每一种扰动方法都会定义一个合成数据分布，每一种局部方法都会定义一个邻域。所得结果解释的是模型在这些设计选择下的行为。

#### **Permutation Importance**

Permutation importance 衡量的是：在评估集中打破某个特征与目标的关联后，预测性能下降了多少。对于损失 $L$、模型 $f$ 与特征 $j$ 的随机排列 $\pi$，

$$
I_j
=
\mathbb E\!\left[
L\!\left(Y,f(X_{-j},X_j^\pi)\right)
\right]
-
\mathbb E[L(Y,f(X))].
$$

如果指标越大越好，减法方向则相反。该方法回答的是：

> 在当前评估分布和排列方案下，这个已拟合模型在多大程度上依赖特征 $j$ 中的信息？

它并不衡量特征的内在价值、因果效应，也不能回答从未收集该特征时重新训练的新模型会有怎样的表现。移除特征后重新训练回答的是另一个问题，因为模型可以学习替代信息。

Permutation importance 应在验证集或测试集上计算，而不是训练集。多次排列可以得到 Monte Carlo 分布。接近零的数值可能意味着特征无用、模型没有使用它、相关特征替代了它，或者评估指标对它影响的错误不敏感。

<div class="diagram-scroll">

![相关特征彼此替代时，逐个排列可能低估其重要性。](assets/correlated-feature-importance.svg){fig-alt="两个相关特征进入模型；排列其中一个后另一个替代特征仍然存在，而分组排列会同时破坏它们共享的信息。"}

</div>

边际排列还会产生自然情况下不会出现的组合。**分组排列（grouped permutation）**可以衡量一组相关特征；**条件排列（conditional permutation）**尝试从给定 $X_{-j}$ 的条件分布中抽取 $X_j$，从而保留依赖关系，但它改变了估计对象，并额外需要一个条件模型。

<details>
<summary><strong>Python：揭示相关替代特征掩盖的重要性</strong></summary>

```python
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(3)
n = 4000
latent = rng.normal(size=n)

# x1 and x2 are noisy measurements of the same latent signal.
x1 = latent + rng.normal(scale=0.12, size=n)
x2 = latent + rng.normal(scale=0.12, size=n)
x3 = rng.normal(size=n)
y = 2.5 * latent + 0.4 * x3 + rng.normal(scale=0.5, size=n)
X = np.column_stack([x1, x2, x3])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, random_state=3
)
model = RandomForestRegressor(
    n_estimators=160, min_samples_leaf=8, random_state=3, n_jobs=1
).fit(X_train, y_train)

baseline = mean_squared_error(y_test, model.predict(X_test))

def permuted_increase(columns, repeats=20):
    increases = []
    for _ in range(repeats):
        changed = X_test.copy()
        order = rng.permutation(len(changed))
        # Use one shared row permutation so grouped features retain their
        # within-row relationship while their link with y is destroyed.
        changed[:, columns] = changed[order][:, columns]
        loss = mean_squared_error(y_test, model.predict(changed))
        increases.append(loss - baseline)
    return np.mean(increases), np.std(increases, ddof=1)

for columns, label in [([0], "x1"), ([1], "x2"), ([2], "x3"), ([0, 1], "{x1, x2}")]:
    mean_increase, sd_increase = permuted_increase(columns)
    print(f"{label:8s}: MSE increase = {mean_increase:.3f} +/- {sd_increase:.3f}")
```

</details>

分组结果不是单个重要性的总和，因为特征可能发生交互并相互替代。除非解释方法在所选输出尺度上显式保证可加性，否则不能把重要性数值当作可相加的贡献预算。

#### **Partial Dependence、ICE 与 ALE**

特征效应图概括了拟合预测随着某个特征变化的方式。

对于特征子集 $S$ 与补集 $C$，经验**部分依赖函数（partial dependence function）**为

$$
\widehat{PD}_S(x_S)
=
\frac{1}{n}\sum_{i=1}^{n}f(x_S,x_C^{(i)}).
$$

它把每一行中的 $X_S$ 都替换为某个网格值，进行预测后取平均。这是在把 $x_S$ 与观测到的 $x_C$ 组合而成的分布上，估计模型的边际响应。如果 $X_S$ 与 $X_C$ 相互依赖，部分组合可能没有数据支持，于是曲线可能概括的是外推行为。

**Individual Conditional Expectation（ICE）**保留每一行：

$$
\widehat{ICE}_i(x_S)=f(x_S,x_C^{(i)}).
$$

PDP 是 ICE 曲线的平均值。大致平行的 ICE 曲线说明效应主要是加性的；曲线交叉或斜率不同则暴露出平均会掩盖的交互与异质行为。Centered ICE 会减去每条曲线在参考点处的数值，从而强调形状而不是基线水平。

**Accumulated Local Effects（ALE）**不进行全局替换。对于一个连续特征，

$$
ALE_j(x)
=
\int_{z_0}^{x}
\mathbb E\!\left[
\frac{\partial f(X)}{\partial x_j}
\middle| X_j=z
\right]dz
-
\text{centering constant}.
$$

实际计算时会把特征范围划分为多个区间。对于自然落在某个区间内的观测，分别在该区间上下边界计算预测并求差，再对局部差异取平均，沿区间累积并中心化。面对相关特征时，ALE 可以减少不合理外推，但稀疏区间、离散变量和拟合较差的模型仍会产生不稳定曲线。

<div class="diagram-scroll">

![PDP、ICE 与 ALE 使用不同的平均方式。](assets/dependence-methods.svg){fig-alt="三个面板比较部分依赖、个体条件期望与累积局部效应，并列出各自的假设与解释。"}

</div>

<details>
<summary><strong>Python：通过模型查询计算 PDP、ICE 与一阶 ALE</strong></summary>

```python
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

rng = np.random.default_rng(11)
n = 2400
x1 = rng.normal(size=n)
x2 = 0.75 * x1 + rng.normal(scale=0.65, size=n)
y = np.sin(1.4 * x1) + 0.7 * x1 * x2 + rng.normal(scale=0.18, size=n)
X = np.column_stack([x1, x2])

model = HistGradientBoostingRegressor(
    max_depth=4, max_iter=180, learning_rate=0.06, random_state=11
).fit(X, y)

grid = np.quantile(x1, np.linspace(0.05, 0.95, 9))

# PDP replaces x1 for every observation and averages predictions.
pdp = []
ice = np.empty((5, len(grid)))
for grid_index, value in enumerate(grid):
    replaced = X.copy()
    replaced[:, 0] = value
    pdp.append(model.predict(replaced).mean())

    local_rows = X[:5].copy()
    local_rows[:, 0] = value
    ice[:, grid_index] = model.predict(local_rows)

# First-order ALE using quantile bins.
edges = np.unique(np.quantile(x1, np.linspace(0, 1, 11)))
bin_ids = np.clip(np.digitize(x1, edges[1:-1]), 0, len(edges) - 2)
local_effects = np.zeros(len(edges) - 1)
bin_counts = np.zeros(len(edges) - 1, dtype=int)

for bin_id in range(len(edges) - 1):
    rows = np.flatnonzero(bin_ids == bin_id)
    bin_counts[bin_id] = len(rows)
    lower = X[rows].copy()
    upper = X[rows].copy()
    lower[:, 0] = edges[bin_id]
    upper[:, 0] = edges[bin_id + 1]
    local_effects[bin_id] = np.mean(model.predict(upper) - model.predict(lower))

ale_at_bins = np.cumsum(local_effects)
ale_at_bins -= np.average(ale_at_bins, weights=bin_counts)

print("grid:", np.round(grid, 2))
print("PDP:", np.round(pdp, 3))
print("first two ICE curves:\n", np.round(ice[:2], 3))
print("ALE bin effects:", np.round(ale_at_bins, 3))
print("observations per ALE bin:", bin_counts.tolist())
```

</details>

PDP、ICE 与 ALE 都是对 $f$ 的描述性探查。曲线上升表示模型分数在该方法定义的输入操作下上升，并不表示干预现实变量会改善结果。图中应展示 rug marks、分位数范围或区间计数，让读者看见模型在哪些区域拥有数据支持。

**对比。**

| 方法 | 范围 | 主要优势 | 主要失效模式 |
|---|---|---|---|
| Permutation importance | 全局 | 与任务指标对齐的模型依赖 | 相关替代特征与不现实排列 |
| PDP | 全局响应 | 简明的平均效应曲线 | 外推与被隐藏的异质性 |
| ICE | 从局部到全局的响应 | 展示个体曲线与交互 | 曲线过多，并继承 PDP 的外推风险 |
| ALE | 全局响应 | 使用观测区域内的局部变化 | 对分箱与稀疏支持敏感 |


#### **LIME 与 SHAP**

**LIME** 通过在实例 $x_0$ 周围采样、查询黑盒模型、按邻近程度加权样本，并拟合一个可解释的局部代理来解释预测：

$$
g^*
=
\arg\min_{g\in G}
\sum_{z\in\mathcal Z}
\pi_{x_0}(z)\bigl(f(z)-g(z)\bigr)^2
+\Omega(g).
$$

$\pi_{x_0}(z)$ 定义局部性，$\Omega(g)$ 惩罚复杂度。解释是拟合出来的代理 $g$，而不是原始模型本身；其有效性仅限于所采样的邻域。核宽度、扰动分布、特征离散方式、稀疏度和随机种子都可能显著改变系数。

一个局部解释至少应报告：

- 被近似的模型输出及其尺度；
- 邻域生成器与距离所使用的表示；
- 核宽度与有效样本量；
- 在加权扰动样本上的局部保真度；
- 对随机种子与合理邻域选择的稳定性。

<details>
<summary><strong>Python：构造并审计一个 LIME 风格的局部代理</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Ridge

rng = np.random.default_rng(23)
X, y = make_moons(n_samples=1800, noise=0.22, random_state=23)
black_box = RandomForestClassifier(
    n_estimators=180, min_samples_leaf=6, random_state=23
).fit(X, y)

x0 = X[np.argmin(np.abs(black_box.predict_proba(X)[:, 1] - 0.55))]

def local_surrogate(kernel_width, seed):
    local_rng = np.random.default_rng(seed)
    perturbations = x0 + local_rng.normal(scale=0.35, size=(2500, 2))
    black_box_scores = black_box.predict_proba(perturbations)[:, 1]
    squared_distance = np.sum((perturbations - x0) ** 2, axis=1)
    weights = np.exp(-squared_distance / (2 * kernel_width**2))

    surrogate = Ridge(alpha=0.05).fit(
        perturbations, black_box_scores, sample_weight=weights
    )
    fitted = surrogate.predict(perturbations)
    weighted_mean = np.average(black_box_scores, weights=weights)
    weighted_sse = np.sum(weights * (black_box_scores - fitted) ** 2)
    weighted_sst = np.sum(weights * (black_box_scores - weighted_mean) ** 2)
    fidelity_r2 = 1 - weighted_sse / weighted_sst
    effective_n = weights.sum() ** 2 / np.sum(weights**2)
    return surrogate.coef_, fidelity_r2, effective_n

print("instance:", np.round(x0, 3))
print("black-box probability:", round(black_box.predict_proba(x0.reshape(1, -1))[0, 1], 3))
for width in [0.15, 0.30, 0.60]:
    coefficient_runs = []
    fidelities = []
    for seed in range(5):
        coefficients, fidelity, effective_n = local_surrogate(width, seed)
        coefficient_runs.append(coefficients)
        fidelities.append(fidelity)
    print(
        f"width={width:.2f}",
        "mean_coef=", np.round(np.mean(coefficient_runs, axis=0), 3),
        "coef_sd=", np.round(np.std(coefficient_runs, axis=0), 3),
        "mean_fidelity=", round(np.mean(fidelities), 3),
        "effective_n=", round(effective_n),
    )
```

</details>

局部保真度高并不表示该邻域在现实中合理或与决策相关。保真度低则说明，即使只看自己生成的局部区域，这个线性解释也不够充分。

**SHAP** 把特征归因与合作博弈中的 Shapley value 联系起来。为已出现特征的集合 $S$ 定义价值函数 $v(S)$，特征 $j$ 的贡献为

$$
\phi_j
=
\sum_{S\subseteq F\setminus\{j\}}
\frac{|S|!(M-|S|-1)!}{M!}
\left[v(S\cup\{j\})-v(S)\right].
$$

这个权重对特征 $j$ 在所有可能进入顺序下的边际贡献取平均。在选定价值函数后，Shapley value 满足：

- **局部准确性或效率（efficiency）：**$f(x)=\phi_0+\sum_j\phi_j$；
- **缺失性或零贡献者：**从不改变价值的特征得到零贡献；
- **对称性：**可以互换的特征得到相同贡献；
- **可加性：**当博弈相加时，解释也相加。

真正困难的步骤不是组合计算，而是定义“特征缺失”的含义。常见的 intervention value function 会用背景数据集中的行替换缺失特征：

$$
v(S)=\mathbb E_{X_{\bar S}}\left[f(x_S,X_{\bar S})\right].
$$

这样做可能打破已出现特征与缺失特征之间的依赖。Conditional SHAP 则在 $P(X_{\bar S}\mid X_S=x_S)$ 下求平均，保留依赖关系，但会以不同方式分配共享信息，并需要额外的条件分布。二者都不是普遍正确的答案；它们回答的是不同问题。

<div class="diagram-scroll">

![SHAP 官方 waterfall plot 通过特征贡献，从背景期望移动到一个模型输出。](assets/shap-waterfall-official.png){fig-alt="一张 SHAP 瀑布图从模型期望输出开始，依次显示正负特征贡献，最终到达一个样本的预测。"}

</div>

*图片来源：[SHAP 官方 waterfall 示例](https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/plots/waterfall.html)，来自采用 [MIT 许可证](https://github.com/shap/shap/blob/master/LICENSE)的 SHAP 项目。图中单位是模型输出单位，并不自动等同于概率百分点。*

<details>
<summary><strong>Python：为三个特征精确计算 interventional Shapley value</strong></summary>

```python
import itertools
import math
import numpy as np

rng = np.random.default_rng(31)
background = rng.normal(size=(800, 3))
instance = np.array([1.2, -0.7, 0.9])

def model(matrix):
    # Deliberately nonlinear, with an interaction between features 0 and 1.
    return (
        0.8 * matrix[:, 0]
        - 0.5 * matrix[:, 1]
        + 1.1 * matrix[:, 0] * matrix[:, 1]
        + np.sin(matrix[:, 2])
    )

features = range(3)

def coalition_value(coalition):
    completed = background.copy()
    for feature in coalition:
        completed[:, feature] = instance[feature]
    return model(completed).mean()

values = {}
for size in range(4):
    for subset in itertools.combinations(features, size):
        values[frozenset(subset)] = coalition_value(subset)

shapley = np.zeros(3)
for feature in features:
    others = [j for j in features if j != feature]
    for size in range(3):
        for subset_tuple in itertools.combinations(others, size):
            subset = frozenset(subset_tuple)
            weight = (
                math.factorial(size)
                * math.factorial(3 - size - 1)
                / math.factorial(3)
            )
            shapley[feature] += weight * (
                values[subset | {feature}] - values[subset]
            )

baseline = values[frozenset()]
prediction = model(instance.reshape(1, -1))[0]
print("background value:", round(baseline, 4))
print("Shapley values:", np.round(shapley, 4))
print("baseline + contributions:", round(baseline + shapley.sum(), 4))
print("model prediction:", round(prediction, 4))
```

</details>

效率检查验证了分解等式，但不能验证背景总体或“缺失特征”语义是否合适。SHAP value 解释的是模型输出，其中也包括虚假或不公平行为；它并不能认证模型本身合理。

**LIME 与 SHAP 对比。**

| 问题 | LIME 风格代理 | SHAP 风格归因 |
|---|---|---|
| 核心对象 | 局部加权的可解释模型 | 可加的特征贡献分配 |
| 主要设计选择 | 邻域与核函数 | 价值函数与背景分布 |
| 保证 | 近似保真度需要经验检验 | 在选定博弈下满足 Shapley 公理 |
| 常见不稳定来源 | 采样与核宽度 | 相关性与背景选择 |
| 最适合的用途 | 用简单代理检查局部形状 | 在明确输出尺度上分解预测 |

#### **反事实解释**

反事实解释搜索一个可行输入 $x'$，使其预测满足目标条件，同时与 $x$ 保持接近：

$$
\min_{x'}
d(x,x')
+\lambda\,\ell\bigl(f(x'),y_{\text{target}}\bigr)
\quad\text{subject to}\quad
x'\in\mathcal F.
$$

$d$ 编码改变成本，$\mathcal F$ 编码可行性。如果没有约束，数学上最近的反事实可能会降低年龄、改变历史事件、修改受保护属性、产生不可能的类别组合，或利用模型错误。有效系统应区分：

- **不可变特征：**决策时的年龄、出身或历史事件；
- **可变但不能直接行动的特征：**信用分数或诊断；
- **可行动特征：**可以控制的付款、文件或配置；
- **方向约束：**债务可以减少，但不能变成负数；
- **因果约束：**改变一个变量时同步更新其下游后果；
- **多样性：**多个本质不同的可行选项可能比一个最优解更有用。

反事实解释并不自动等于**补救（recourse）**。补救要求当事人能够执行该行动、机构之后仍采用同一政策，而且行动很可能产生预期的现实结果。模型反事实只说明 $f(x')$ 发生变化。

<details>
<summary><strong>Python：搜索一个受约束且可审计的反事实</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(41)
n = 5000
income = rng.normal(65, 18, size=n)       # thousands per year
debt = np.clip(rng.normal(24, 12, size=n), 0, None)
age = rng.integers(21, 66, size=n)
X = np.column_stack([income, debt, age])

latent_score = 0.065 * income - 0.095 * debt + 0.012 * age - 3.2
y = rng.binomial(1, 1 / (1 + np.exp(-latent_score)))

scaler = StandardScaler().fit(X)
model = LogisticRegression(max_iter=2000).fit(scaler.transform(X), y)

# Select a person just below the operational threshold.
probabilities = model.predict_proba(scaler.transform(X))[:, 1]
eligible = np.flatnonzero((probabilities > 0.30) & (probabilities < 0.45))
index = eligible[np.argmax(probabilities[eligible])]
x = X[index].copy()
starting_probability = probabilities[index]

candidates = []
for income_increase in np.linspace(0, 30, 31):
    for debt_reduction in np.linspace(0, min(25, x[1]), 26):
        candidate = x.copy()
        candidate[0] += income_increase       # income can only increase
        candidate[1] -= debt_reduction        # debt can only decrease
        candidate[2] = x[2]                   # age is immutable
        probability = model.predict_proba(
            scaler.transform(candidate.reshape(1, -1))
        )[0, 1]
        if probability >= 0.50:
            # Domain-specific cost: one unit of debt reduction is treated as
            # more burdensome than one unit of additional annual income.
            cost = income_increase / 10 + debt_reduction / 5
            candidates.append((cost, probability, candidate))

cost, new_probability, counterfactual = min(candidates, key=lambda row: row[0])
print("original [income, debt, age]:", np.round(x, 2))
print("original probability:", round(starting_probability, 3))
print("counterfactual:", np.round(counterfactual, 2))
print("counterfactual probability:", round(new_probability, 3))
print("action cost:", round(cost, 3))
```

</details>

优化结果的可辩护程度不会超过约束与成本函数本身。应跨模型版本对反事实进行压力测试，检查不同群体在可行性和成本上的差异，并把它与“推荐行动会导致期望结果”的因果主张明确区分。

**总结。**LIME 近似局部形状，SHAP 在选定的 coalition game 下分配预测贡献，反事实则在约束下搜索能改变决策的输入。它们都不能独立证明因果补救、公平性或模型正确性。


### **鲁棒性与分布偏移**

鲁棒性是指系统在偏离标称条件的指定范围内，仍能保持可接受行为的能力。如果没有以下三个要素，“鲁棒”这个词就是不完整的：

1. 描述哪些因素可以改变的**扰动或偏移集合**；
2. 描述哪些行为必须保持可接受的**性能要求**；
3. 描述检测、拒绝预测、回退、恢复与监控的**运行响应**。

模型可以对 Gaussian sensor noise 很鲁棒，却对缺失字段非常脆弱；可以在一家医院内稳定，却在另一家医院中不安全；可以抵抗细微图像扰动，却容易受到数据投毒。鲁棒性永远是相对于威胁模型或环境模型而言的。

#### **噪声、破坏与压力测试**

令 $P_{\text{train}}(X,Y)$ 为训练数据所代表的数据生成分布，$P_{\text{deploy}}(X,Y)$ 为部署分布。常见偏移包括：

- **Covariate shift：**$P(X)$ 变化，同时假设 $P(Y\mid X)$ 保持稳定。
- **Label shift：**$P(Y)$ 变化，同时假设 $P(X\mid Y)$ 保持稳定。
- **Concept shift：**$P(Y\mid X)$ 变化，旧的预测关系不再稳定。
- **Subgroup 或 mixture shift：**环境或群体的比例变化，或某个群体内部性能变化。
- **Support shift：**部署数据进入训练支持很少或完全没有支持的区域。
- **Measurement shift：**现实构念近似相同，但传感器、编码规则、缺失机制或预处理不同。

<div class="diagram-scroll">

![不同分布偏移会改变数据生成过程中的不同部分。](assets/distribution-shift-taxonomy.svg){fig-alt="四个面板比较 covariate、label、concept 与 subgroup shift，并说明它们需要不同的假设和响应。"}

</div>

这些标签不是可以直接观察到的事实。把问题称为 covariate shift，等于声称条件机制保持稳定。这个假设应由领域知识支持，并在延迟标签最终到达后进行检验。

**压力测试（stress test）**在一系列严重程度下施加合理破坏，并且不能只测平均准确率：

- 任务损失、校准与拒绝预测率；
- 最差群体与较低分位性能；
- 对缺失模式与 schema 变化的敏感性；
- 随严重程度单调退化，而不是只测试一个任意破坏水平；
- 置信区间与多次随机破坏；
- 运行延迟、资源耗尽与回退是否成功。

破坏方式应反映真实部署过程。Gaussian noise 可以作为单元测试，但很少能代表所有现实故障。对于表格数据，应测试裁剪、舍入、单位错误、陈旧字段、sentinel value、missing-not-at-random 模式、类别漂移与重复记录。对于文本，应测试拼写错误、code switching、否定、方言、prompt injection 与截断。对于图像和音频，应测试模糊、压缩、光照、遮挡、设备变化与时间伪影。

<details>
<summary><strong>Python：构造包含严重程度与群体报告的压力测试矩阵</strong></summary>

```python
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(53)
n = 6000
X = rng.normal(size=(n, 8))
true_logit = (
    1.45 * X[:, 0]
    - 1.20 * X[:, 1]
    + 1.05 * X[:, 2]
    + 0.75 * X[:, 3]
    - 0.35 * X[:, 4]
)
true_probability = 1 / (1 + np.exp(-true_logit))
y = rng.binomial(1, true_probability)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=53
)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=2000),
).fit(X_train, y_train)

# This is an audit slice, not a claim that the feature defines a social group.
group = X_test[:, 1] > np.median(X_train[:, 1])

def evaluate(name, changed):
    probability = model.predict_proba(changed)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    group_accuracies = [
        accuracy_score(y_test[group == value], prediction[group == value])
        for value in [False, True]
    ]
    return {
        "scenario": name,
        "accuracy": accuracy_score(y_test, prediction),
        "log_loss": log_loss(y_test, probability),
        "worst_group_accuracy": min(group_accuracies),
        "group_gap": abs(group_accuracies[0] - group_accuracies[1]),
    }

results = [evaluate("clean", X_test)]
for severity in [0.15, 0.35, 0.70]:
    noisy = X_test + rng.normal(scale=severity, size=X_test.shape)
    results.append(evaluate(f"noise-{severity:.2f}", noisy))

for missing_rate in [0.05, 0.15, 0.30]:
    missing = X_test.copy()
    mask = rng.random(missing.shape) < missing_rate
    missing[mask] = np.nan
    results.append(evaluate(f"missing-{missing_rate:.2f}", missing))

for row in results:
    print(
        f"{row['scenario']:14s}",
        f"acc={row['accuracy']:.3f}",
        f"logloss={row['log_loss']:.3f}",
        f"worst_group={row['worst_group_accuracy']:.3f}",
        f"gap={row['group_gap']:.3f}",
    )
```

</details>

最差群体指标可能比总体指标退化得更快。部署门槛应关联明确容差，例如“在预期缺失率第 95 百分位之前，最差切片 recall 始终高于 0.82”，而不是笼统地声称退化“很小”。

鲁棒性干预可以发生在多个层次：

| 层次 | 示例 | 限制 |
|---|---|---|
| 数据 | 更完整覆盖、数据增强、破坏模拟、重新加权 | 合成扰动可能不匹配部署 |
| 模型 | 正则化、不变特征、鲁棒损失、对抗训练 | 可能用干净数据效用换取狭窄的鲁棒目标 |
| 不确定性 | 校准、ensemble、conformal set、OOD score | 不确定性在偏移下也可能失效 |
| 系统 | 验证、schema contract、回退、人工升级、回滚 | 需要明确运行负责人并经过测试 |
| 监控 | 漂移、切片指标、延迟标签、事件复盘 | 检测并不等于纠正 |

#### **对抗样本**

**对抗样本（adversarial example）**是为了造成模型错误而被有意修改的输入，同时修改必须满足指定扰动约束。对于参数为 $\theta$、损失为 $\ell$、输入为 $x$、标签为 $y$ 的分类器，untargeted evasion attack 求解

$$
\max_{\delta\in\Delta}
\ell\bigl(f_\theta(x+\delta),y\bigr),
$$

其中 $\Delta$ 可以约束 $\|\delta\|_\infty\le\epsilon$、保持语义有效、限制可编辑字段，或模拟物理变换。Targeted attack 则把输出推向攻击者选定的类别。

**Fast Gradient Sign Method（FGSM）**执行一步：

$$
x_{\text{adv}}
=
\operatorname{clip}\left(
x+\epsilon\,\operatorname{sign}
\left(\nabla_x\ell(f_\theta(x),y)\right)
\right).
$$

Projected gradient descent 会重复较小步长，并把结果投影回 $\Delta$。只有威胁模型明确以下内容后，这些公式才真正有意义：

- 攻击目标：evasion、poisoning、extraction 或 inference；
- 攻击能力：可以改变哪些输入、训练样本、标签或查询；
- 攻击知识：white-box 参数、梯度、架构、数据，或 black-box 输出；
- 攻击预算：范数、字段数量、语义约束、查询次数或物理成本；
- 成功标准与防御假设。

<div class="diagram-scroll">

![Adversarial Robustness Toolbox 区分 evasion、poisoning、extraction 与 inference 威胁。](assets/art-adversarial-threats-official.png){fig-alt="一张四块拼图把 evasion、poisoning、extraction 和 inference 标为对抗机器学习威胁类别。"}

</div>

*图片来源：[Adversarial Robustness Toolbox 威胁图](https://github.com/Trusted-AI/adversarial-robustness-toolbox/blob/main/docs/images/adversarial_threats_art.png)，采用 [MIT 许可证](https://github.com/Trusted-AI/adversarial-robustness-toolbox/blob/main/LICENSE)。*

<details>
<summary><strong>Python：对可微逻辑分类器实现 FGSM</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=5000,
    n_features=12,
    n_informative=8,
    class_sep=1.3,
    random_state=61,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=61
)

scaler = StandardScaler().fit(X_train)
X_train_z = scaler.transform(X_train)
X_test_z = scaler.transform(X_test)
model = LogisticRegression(max_iter=2000).fit(X_train_z, y_train)

weight = model.coef_[0]
bias = model.intercept_[0]

def sigmoid(value):
    value = np.clip(value, -40, 40)
    return 1 / (1 + np.exp(-value))

def predict(matrix):
    return (sigmoid(matrix @ weight + bias) >= 0.5).astype(int)

def fgsm(matrix, labels, epsilon):
    probability = sigmoid(matrix @ weight + bias)
    # Gradient of binary cross-entropy with respect to standardized input.
    gradient = (probability - labels)[:, None] * weight[None, :]
    return matrix + epsilon * np.sign(gradient)

print("clean accuracy:", round(accuracy_score(y_test, predict(X_test_z)), 3))
for epsilon in [0.02, 0.05, 0.10, 0.20]:
    adversarial = fgsm(X_test_z, y_test, epsilon)
    linf = np.max(np.abs(adversarial - X_test_z), axis=1).mean()
    print(
        f"epsilon={epsilon:.2f}",
        f"mean_Linf={linf:.3f}",
        f"accuracy={accuracy_score(y_test, predict(adversarial)):.3f}",
    )
```

</details>

该攻击与标准化后的逻辑模型完全对齐，因此它是透明的演示，而不是现实安全认证。对于类别型或受约束表格数据，$L_p$ 球可能允许不可能的记录。对于文本，同义词替换可能改变语义；对于图像，像素范数并不保证感知等价。应评估与真实接口匹配的攻击，并面对了解防御机制的自适应攻击者。

对抗训练近似求解

$$
\min_\theta
\mathbb E_{(X,Y)}
\left[
\max_{\delta\in\Delta}
\ell(f_\theta(X+\delta),Y)
\right].
$$

它可以改善训练威胁集合内的性能，但鲁棒性未必能迁移到不同范数、预算、攻击、破坏或分布。Gradient masking 可能让较弱攻击看似失败，却没有真正提高安全性；应使用多种强攻击、攻击重启、sanity check 与独立评估。

#### **分布外检测**

分布外（out-of-distribution, OOD）检测器尝试识别超出模型已验证分布的输入。这需要相对于训练或部署参考来定义“外部”。Near-OOD 样本共享低层结构，却在重要类别或环境上不同；far-OOD 样本在视觉或统计上可能很明显。

常见分数包括：

- 最大 softmax 或类别概率；
- 预测熵或 ensemble disagreement；
- 输入或表示空间中的距离；
- 生成模型下的密度或 likelihood；
- energy score、one-class model 与重构误差。

没有任何分数普遍可靠。判别分类器可以在远离训练支持的位置仍然高度自信，likelihood model 也可能给无关但简单的输入分配高密度。应在多种现实 OOD 来源上评估，并报告 AUROC、precision-recall 曲线下面积、目标 true-positive rate 下的 false-positive rate，以及运行阈值的不确定性。

OOD 检测经常与**选择性预测（selective prediction）**结合。仅当置信分数 $s(x)$ 超过阈值 $\tau$ 时，模型才给出预测：

$$
\operatorname{coverage}(\tau)
=P(s(X)\ge\tau),
$$

$$
\operatorname{risk}(\tau)
=
\mathbb E\!\left[
L(Y,f(X))
\mid s(X)\ge\tau
\right].
$$

Risk-coverage curve 展示系统增加拒绝预测后，已接受案例中的错误如何变化。

<div class="diagram-scroll">

![Risk-coverage curve 明确展示拒绝预测的权衡。](assets/risk-coverage-curve.svg){fig-alt="一条曲线显示 conditional risk 随 coverage 增大而上升，并标记一个平衡已接受案例与拒绝预测的阈值。"}

</div>

<details>
<summary><strong>Python：比较 OOD 分数并构造 risk-coverage 表</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(71)

# Two in-distribution classes.
class_0 = rng.normal(loc=[-1.2, 0.0], scale=[0.75, 0.85], size=(1500, 2))
class_1 = rng.normal(loc=[1.2, 0.0], scale=[0.75, 0.85], size=(1500, 2))
X_train = np.vstack([class_0[:1000], class_1[:1000]])
y_train = np.r_[np.zeros(1000, dtype=int), np.ones(1000, dtype=int)]
X_test = np.vstack([class_0[1000:], class_1[1000:]])
y_test = np.r_[np.zeros(500, dtype=int), np.ones(500, dtype=int)]

# OOD data lie above both classes but can still receive confident class scores.
X_ood = rng.normal(loc=[0.0, 4.0], scale=[1.1, 0.65], size=(1000, 2))

scaler = StandardScaler().fit(X_train)
train_z = scaler.transform(X_train)
test_z = scaler.transform(X_test)
ood_z = scaler.transform(X_ood)

model = LogisticRegression(max_iter=2000).fit(train_z, y_train)
test_probability = model.predict_proba(test_z)
ood_probability = model.predict_proba(ood_z)

confidence_test = test_probability.max(axis=1)
confidence_ood = ood_probability.max(axis=1)
confidence_ood_score = 1 - np.r_[confidence_test, confidence_ood]

neighbors = NearestNeighbors(n_neighbors=10).fit(train_z)
distance_test = neighbors.kneighbors(test_z, return_distance=True)[0].mean(axis=1)
distance_ood = neighbors.kneighbors(ood_z, return_distance=True)[0].mean(axis=1)
distance_ood_score = np.r_[distance_test, distance_ood]

ood_label = np.r_[np.zeros(len(test_z)), np.ones(len(ood_z))]
print("OOD AUROC from low confidence:", round(roc_auc_score(ood_label, confidence_ood_score), 3))
print("OOD AUROC from neighbor distance:", round(roc_auc_score(ood_label, distance_ood_score), 3))

# Risk-coverage on labeled in-distribution test data.
prediction = test_probability.argmax(axis=1)
error = (prediction != y_test).astype(float)
order = np.argsort(-confidence_test)
for coverage in [0.25, 0.50, 0.75, 1.00]:
    accepted = order[: max(1, int(coverage * len(order)))]
    threshold = confidence_test[accepted].min()
    print(
        f"coverage={coverage:.2f}",
        f"threshold={threshold:.3f}",
        f"selective_risk={error[accepted].mean():.3f}",
    )
```

</details>

在这个刻意构造的几何示例中，距离非常有效，但在高维原始空间中可能失去意义。表示选择是检测器的一部分，阈值应根据部署成本确定：拒绝过多案例可能压垮人工审核，或系统性地延迟与拒绝服务。

**总结。**鲁棒性要求明确偏移或攻击者、严重程度范围、切片感知指标与经过测试的响应。压力测试、对抗评估、OOD 检测与拒绝预测相互补充。它们都无法替代带延迟真值标签的监控，因为最重要的 concept shift 可能无法仅从 $X$ 中观察出来。


### **机器学习公平性**

公平性不是一种可以脱离应用单独选择的统计属性。它是一个规范性与社会技术问题，关心的是：**哪些人在怎样的决策过程中面临哪些伤害，以及他们是否有机会获得复核或补救**。定量指标可以帮助检验已经声明的关切，却不能替人决定哪一种关切最重要。

公平性分析应明确：

1. 被评估的决策、分数、排序或资源分配；
2. 受影响的人、相关群体与交叉群体；
3. 收益或伤害，包括 false positive、false negative、延迟、服务质量与机会被拒绝；
4. 时间范围与反馈循环；
5. 公平准则，以及它为什么与该伤害匹配；
6. 不确定性、样本支持与实际有意义的容差；
7. 谁负责缓解决策，以及受影响者如何提出申诉。

#### **偏差来源**

偏差可以在模型拟合之前、期间与之后进入：

| 阶段 | 机制 | 示例诊断 |
|---|---|---|
| 问题设定 | 目标或决策编码了不公正政策 | 比较声明的目标与真实机构目的 |
| 抽样 | 某些群体缺失或被选择性观察 | 按群体检查覆盖、无响应与支持 |
| 测量 | 特征或标签对不同群体具有不同误差 | 按群体检查标注者一致性与标签效度 |
| 历史过程 | 标签复制过去的资源分配或执法 | 审计标签如何生成，以及谁曾暴露于该过程 |
| 表示 | 特征包含代理，或丢失相关背景 | 条件误差与表示距离 |
| 学习 | 平均损失牺牲较小或更困难的群体 | 分群体学习曲线与最差群体风险 |
| 阈值 | 一个分数阈值产生不同后果 | 分群体混淆率与效用 |
| 部署 | 用户适应，自动化改变行为，申诉机会不同 | 流程指标、人工覆盖、延迟与长期结果 |

移除受保护属性并不能保证公平。其他变量可以成为它的 proxy，而且为了测量差异、学习群体特定的测量误差，或实施在法律和伦理上合理的缓解措施，可能仍然需要该属性。反过来，使用该属性也可能引入新风险。访问权限、使用目的与治理必须明确。

公平性应在**决策总体**上测量，而不能只看恰好拥有标签的行。选择性标签十分常见：只有获批申请人才会被观察到贷款偿还情况，只有接受治疗的患者才有治疗结果，而再犯可能通过不平等的监控被测量。标准测试指标此时可能以一个有偏的观察过程为条件。

#### **Demographic Parity 与 Equalized Odds**

令受保护或审计群体为 $A$，真实标签为 $Y$，预测为 $\hat Y$，分数为 $S$。

**Demographic parity** 要求正向决策率相等：

$$
P(\hat Y=1\mid A=a)
=
P(\hat Y=1\mid A=b).
$$

当资源分配比例本身就是关切，或标签不可信时，这个准则具有意义；但它忽略标签，并且在 base rate 不同时可能需要截然不同的错误率。

**Equal opportunity** 要求 true-positive rate 相等：

$$
P(\hat Y=1\mid Y=1,A=a)
=
P(\hat Y=1\mid Y=1,A=b).
$$

它关注标签为正的人获得机会的情况。**Equalized odds** 进一步要求 false-positive rate 也相等：

$$
\hat Y\perp A\mid Y.
$$

等价地，

$$
TPR_a=TPR_b
\quad\text{and}\quad
FPR_a=FPR_b.
$$

哪一种错误更重要取决于应用。在安全警报中，false negative 可能漏掉危险；在指控系统中，false positive 可能造成严重伤害。Equalized odds 把两种错误率差异都视为相关，却没有编码它们的相对成本。

**Predictive parity** 要求 positive predictive value 相等：

$$
P(Y=1\mid \hat Y=1,A=a)
=
P(Y=1\mid \hat Y=1,A=b).
$$

它询问正向决策在不同群体中是否具有相同含义。这与以真实标签为条件的 equalized odds 不同。

<div class="diagram-scroll">

![常见公平准则施加不同的条件独立关系。](assets/fairness-criteria-map.svg){fig-alt="四个面板分别定义 demographic parity、equalized odds、predictive parity 与 calibration，并提醒在 base rate 不同时这些准则通常互相冲突。"}

</div>

<details>
<summary><strong>Python：审计选择率、错误率与预测值差异</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(83)
n = 12_000
group = rng.binomial(1, 0.45, size=n)
x = rng.normal(loc=0.25 * group, scale=1.0, size=n)

# Different group base rates and measurement quality produce several kinds
# of disparity even though the same threshold is used.
true_logit = -0.8 + 1.25 * x + 0.65 * group
true_probability = 1 / (1 + np.exp(-true_logit))
y = rng.binomial(1, true_probability)

score_logit = -0.65 + 1.05 * x + 0.15 * group + rng.normal(
    scale=0.45 + 0.25 * group, size=n
)
score = 1 / (1 + np.exp(-score_logit))
prediction = (score >= 0.50).astype(int)

def safe_ratio(numerator, denominator):
    return numerator / denominator if denominator else np.nan

def group_metrics(group_value):
    mask = group == group_value
    truth = y[mask]
    pred = prediction[mask]
    tp = np.sum((truth == 1) & (pred == 1))
    fp = np.sum((truth == 0) & (pred == 1))
    tn = np.sum((truth == 0) & (pred == 0))
    fn = np.sum((truth == 1) & (pred == 0))
    return {
        "n": mask.sum(),
        "base_rate": truth.mean(),
        "selection_rate": pred.mean(),
        "tpr": safe_ratio(tp, tp + fn),
        "fpr": safe_ratio(fp, fp + tn),
        "ppv": safe_ratio(tp, tp + fp),
    }

metrics = [group_metrics(value) for value in [0, 1]]
for value, row in enumerate(metrics):
    display = {
        key: int(val) if key == "n" else round(float(val), 3)
        for key, val in row.items()
    }
    print("group", value, display)

print("\nabsolute gaps")
for key in ["selection_rate", "tpr", "fpr", "ppv"]:
    print(key, round(abs(metrics[0][key] - metrics[1][key]), 3))
```

</details>

报告比例时也要报告分母。由 20 个负样本支持的 10 个百分点 false-positive gap，与由 20,000 个样本支持的同样差距有完全不同的不确定性。对于排序系统，应检查曝光与 position-weighted utility，而不是把所有输出强行转换成二元分类。

#### **校准与个体公平**

如果对于相关分数区间和每个群体 $a$ 都满足

$$
P(Y=1\mid S=s,A=a)=s,
$$

则概率分数在群体内**经过校准（calibrated within groups）**。实际中通常用分箱或平滑校准曲线估计，并同时报告不确定性。平均 calibration error 可能掩盖严重的局部错误，尤其是在样本稀疏的高风险区域。

当群体 base rate 不同而预测又不完美时，多种看似合理的准则通常无法同时满足。一个分数可以在每个群体中都校准良好，却在共同阈值下产生不同的 false-positive 与 true-positive rate。强制 equalized odds 可能需要按群体随机化决策，从而不再保留相同的分数语义。这不是软件错误，而是因为这些准则编码了不同的公平观念。

<details>
<summary><strong>Python：展示经过群体校准却不满足 equalized odds 的分数</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(89)
n = 100_000
group = rng.binomial(1, 0.5, size=n)

# Scores come from different beta distributions, so group base rates differ.
# Drawing Y ~ Bernoulli(score) makes the score calibrated by construction.
score = np.empty(n)
score[group == 0] = rng.beta(2.0, 5.0, size=np.sum(group == 0))
score[group == 1] = rng.beta(5.0, 2.5, size=np.sum(group == 1))
y = rng.binomial(1, score)
prediction = score >= 0.5

def expected_calibration_error(mask, bins=12):
    edges = np.linspace(0, 1, bins + 1)
    total = mask.sum()
    ece = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        in_bin = mask & (score >= left) & (
            (score < right) if right < 1 else (score <= right)
        )
        if in_bin.any():
            ece += in_bin.sum() / total * abs(y[in_bin].mean() - score[in_bin].mean())
    return ece

for value in [0, 1]:
    mask = group == value
    positives = mask & (y == 1)
    negatives = mask & (y == 0)
    tpr = prediction[positives].mean()
    fpr = prediction[negatives].mean()
    ppv = y[mask & prediction].mean()
    print(
        f"group={value}",
        f"base_rate={y[mask].mean():.3f}",
        f"ECE={expected_calibration_error(mask):.4f}",
        f"TPR={tpr:.3f}",
        f"FPR={fpr:.3f}",
        f"PPV={ppv:.3f}",
    )
```

</details>

接近零的 calibration error 与不同的 TPR、FPR、PPV 同时存在。因此，指标选择需要围绕伤害、标签效度与决策含义进行实质论证，不能通过挑选指标完成。

**个体公平（individual fairness）**常表述为“相似个体应受到相似对待”：

$$
d_Y\bigl(f(x_i),f(x_j)\bigr)
\le
L\,d_X(x_i,x_j).
$$

数学不等式本身很直接，困难的是定义任务特定且公平的距离 $d_X$。历史特征可能编码不公正，而在记录数据中相似的两个人可能面对不同现实约束。**Counterfactual fairness** 则询问：在受保护属性不同、适当背景因素保持不变的反事实世界中，决策是否仍保持相同。它需要因果模型，并继承其中不可检验的假设。

群体公平与个体公平可能冲突。一个平滑且对个体一致的规则可能保留数据中继承的群体差异，而群体平等干预可能对阈值附近原本相似的人做出不同决策。应记录这种冲突，而不是把它隐藏在一个复合分数中。

#### **预处理、训练中与后处理缓解**

公平性缓解可以修改数据、学习过程或最终决策。

<div class="diagram-scroll">

![公平缓解可以发生在模型训练之前、期间或之后。](assets/fairness-mitigation-pipeline.svg){fig-alt="一个三阶段流水线比较 preprocessing、in-processing 与 post-processing 公平干预及其限制。"}

</div>

**Pre-processing** 方法重新加权、重采样、重新标注或变换数据。它们可以保留原学习器不变，但可能让下游使用者看不到干预，而且无法修复无效目标。**In-processing** 方法在训练中加入公平约束、robust group objective 或 adversarial representation loss。它们直接优化所选准则，但需要访问训练内部，并仔细检查公平约束能否泛化。**Post-processing** 在拟合后改变阈值或随机化决策。它便于复用已有分数，但可能需要在决策时获取群体信息，也可能降低分数语义的一致性。

对于群体特定阈值 $t_a$，一个约束形式为

$$
\min_{\{t_a\}}
\widehat R(\{t_a\})
\quad\text{subject to}\quad
\max_{a,b}
\left\{
|TPR_a-TPR_b|,
|FPR_a-FPR_b|
\right\}
\le \epsilon.
$$

$\epsilon$ 是明确容差，并不是神奇的公平定义。阈值应在验证数据上拟合，再在未接触测试集上评估一次；在同一个样本上优化并报告会夸大平等程度。

<details>
<summary><strong>Python：搜索群体阈值并揭示效用—平等权衡</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(97)
n = 18_000
group = rng.binomial(1, 0.45, size=n)
ability = rng.normal(size=n)
true_probability = 1 / (1 + np.exp(-(-0.4 + 1.35 * ability + 0.45 * group)))
y = rng.binomial(1, true_probability)

# Group 1 receives a noisier score.
raw_score = -0.35 + 1.15 * ability + rng.normal(
    scale=0.55 + 0.45 * group, size=n
)
score = 1 / (1 + np.exp(-raw_score))

def evaluate(threshold_0, threshold_1):
    threshold = np.where(group == 0, threshold_0, threshold_1)
    prediction = score >= threshold
    tpr, fpr = [], []
    for value in [0, 1]:
        mask = group == value
        tpr.append(prediction[mask & (y == 1)].mean())
        fpr.append(prediction[mask & (y == 0)].mean())
    error = np.mean(prediction != y)
    equalized_odds_gap = max(abs(tpr[0] - tpr[1]), abs(fpr[0] - fpr[1]))
    return error, equalized_odds_gap, tpr, fpr

baseline = evaluate(0.5, 0.5)
candidate_rows = []
for threshold_0 in np.linspace(0.20, 0.80, 31):
    for threshold_1 in np.linspace(0.20, 0.80, 31):
        metrics = evaluate(threshold_0, threshold_1)
        candidate_rows.append((metrics[0] + 1.2 * metrics[1], threshold_0, threshold_1, metrics))

_, best_t0, best_t1, mitigated = min(candidate_rows, key=lambda row: row[0])
print(
    "shared threshold:",
    f"error={baseline[0]:.3f}",
    f"EO_gap={baseline[1]:.3f}",
    "TPR=", np.round(baseline[2], 3),
    "FPR=", np.round(baseline[3], 3),
)
print(
    f"group thresholds: t0={best_t0:.2f}, t1={best_t1:.2f}",
    f"error={mitigated[0]:.3f}",
    f"EO_gap={mitigated[1]:.3f}",
    "TPR=", np.round(mitigated[2], 3),
    "FPR=", np.round(mitigated[3], 3),
)
```

</details>

惩罚权重选择的是 utility-parity frontier 上的一个点。不同社会成本、容差或群体定义会选择另一个点。应报告整条 frontier 与决策规则，而不是把选中的模型描述成客观意义上的“已消除偏差”。

对于样本较小的交叉群体，公平性的泛化尤其脆弱。一个指标在完整测试集上看似接近，在 bootstrap 样本中却可能大幅变化，或在年龄、性别、地点、残障、语言与设备可访问性的交叉处失效。

<details>
<summary><strong>Python：量化交叉群体 recall 的不确定性</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(101)
n = 9000
group_a = rng.binomial(1, 0.35, size=n)
group_b = rng.binomial(1, 0.25, size=n)
signal = rng.normal(size=n)

true_probability = 1 / (1 + np.exp(-(-0.5 + 1.25 * signal)))
y = rng.binomial(1, true_probability)

# The small intersection receives both noisier and systematically lower scores.
intersection = (group_a == 1) & (group_b == 1)
noise_scale = np.where(intersection, 1.35, 0.55)
score = 1 / (
    1
    + np.exp(
        -(
            -0.45
            + signal
            - 0.85 * intersection
            + rng.normal(scale=noise_scale)
        )
    )
)
prediction = score >= 0.5

def recall(indices):
    positives = indices[y[indices] == 1]
    return prediction[positives].mean() if len(positives) else np.nan

for a in [0, 1]:
    for b in [0, 1]:
        indices = np.flatnonzero((group_a == a) & (group_b == b))
        point = recall(indices)
        bootstrap = []
        for _ in range(500):
            sample = rng.choice(indices, size=len(indices), replace=True)
            value = recall(sample)
            if not np.isnan(value):
                bootstrap.append(value)
        lower, upper = np.quantile(bootstrap, [0.025, 0.975])
        print(
            f"A={a}, B={b}",
            f"n={len(indices):4d}",
            f"recall={point:.3f}",
            f"95% bootstrap interval=({lower:.3f}, {upper:.3f})",
        )
```

</details>

还需要考虑多重比较：搜索数百个切片后只报告最大差距，必然会找到噪声。应预先声明关键切片，对探索性搜索控制 false discovery，并与领域专家及受影响群体一起调查持续存在的差异。

**总结。**公平指标是对已声明伤害模型的检验，不是通用认证。应报告 base rate、混淆矩阵组成、样本量、不确定性、交叉群体与效用，并评估完整决策过程，包括谁能获得标签、谁能够申诉，以及系统将如何改变未来数据。


### **隐私**

当有关个人或组织的数据可以在预期用途之外被获知时，就会产生隐私风险。删除姓名通常远远不够。Quasi-identifier 可以重新识别记录，模型参数可以记忆稀有样本，梯度可以泄露训练内容，多次查询也可以累积证据。

隐私分析从一个**威胁模型**开始：

- **隐私单元：**一行、一个人、家庭、设备、客户端、事件，或属于同一个人的全部记录。
- **受保护信息：**参与身份、某个属性、原始内容、关系、模型更新或总体统计量。
- **攻击者：**模型使用者、服务器、其他客户端、内部人员、数据接收者或外部观察者。
- **访问能力：**标签、概率、embedding、梯度、参数、时间信息或重复自适应查询。
- **辅助知识：**公共记录、部分特征、shadow data 或训练算法知识。
- **发布历史：**之前的模型、dashboard、checkpoint 与相关数据集。

<div class="diagram-scroll">

![隐私泄漏可能穿过数据、模型参数、接口与辅助信息。](assets/privacy-threat-surface.svg){fig-alt="一条从训练数据经过模型和接口到达攻击者的流水线，列出 membership inference、梯度泄漏、自适应查询与辅助数据。"}

</div>

数据最小化、保留期限、加密、身份认证、访问控制、日志与事件响应仍然不可缺少。数学隐私保证只控制定义好的信息通道；它无法修复暴露的数据库、恶意客户端、存在漏洞的 endpoint，或模型之外的 side channel。

#### **成员与属性推断**

**Membership inference attack** 预测目标记录是否参与了训练。过拟合模型经常在训练记录上产生更低损失或更高置信度，为攻击者提供信号。攻击可以使用目标标签、概率向量、逐样本损失、梯度或重复的数据增强查询。

攻击性能应在现实的成员先验和运行点下衡量。AUROC 概括排序能力，但在成员只占总体极小比例时，可能掩盖很高的 false-positive rate。应报告部署相关先验下的 precision 或 advantage，并包含有竞争力的简单基线。

<details>
<summary><strong>Python：把泛化差距转化为 membership signal</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(
    n_samples=4000,
    n_features=80,
    n_informative=16,
    n_redundant=8,
    flip_y=0.08,
    random_state=107,
)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, train_size=700, test_size=700, stratify=y, random_state=107
)

models = {
    "unpruned tree": DecisionTreeClassifier(random_state=107),
    "regularized logistic": LogisticRegression(C=0.15, max_iter=3000),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    train_probability = model.predict_proba(X_train)
    holdout_probability = model.predict_proba(X_holdout)

    # Attacker knows each target's true label and uses confidence assigned to it.
    train_confidence = train_probability[np.arange(len(y_train)), y_train]
    holdout_confidence = holdout_probability[np.arange(len(y_holdout)), y_holdout]
    attack_score = np.r_[train_confidence, holdout_confidence]
    membership = np.r_[np.ones(len(y_train)), np.zeros(len(y_holdout))]

    print(
        name,
        f"train_acc={accuracy_score(y_train, model.predict(X_train)):.3f}",
        f"test_acc={accuracy_score(y_holdout, model.predict(X_holdout)):.3f}",
        f"membership_AUROC={roc_auc_score(membership, attack_score):.3f}",
    )
```

</details>

正则化与良好泛化可以削弱这种简单攻击，但不能提供最坏情况隐私保证。即使平均训练和测试损失相同，稀有记录或重复记录仍可能容易受到攻击。

**Attribute inference attack** 使用已知属性与模型访问来推断隐藏的敏感属性。**Model inversion 或 reconstruction attack** 尝试恢复具有代表性的训练内容或具体训练记录。**Model extraction** 通过查询重构模型行为或参数；它通常涉及知识产权或安全，也可以支持更强的隐私攻击。这些类别存在重叠，因此评估应跟随攻击者的真实目标，而不能只依赖 taxonomy 名称。

防御措施包括最小化输出、对置信度舍入、rate limit、审计日志、正则化、early stopping、数据去重、访问控制、private aggregation 与 differential privacy。输出扰动可能降低某一种攻击，却在自适应查询下继续泄漏；应由了解防御机制的攻击者进行测试。

#### **差分隐私**

差分隐私（differential privacy, DP）限制的是：加入、删除或替换一个受保护单元后，算法输出分布最多可以改变多少。如果对于每一对相邻数据集 $D\sim D'$ 和每个可测输出集合 $S$ 都满足

$$
P\bigl(\mathcal M(D)\in S\bigr)
\le
e^\epsilon
P\bigl(\mathcal M(D')\in S\bigr)
+\delta,
$$

则随机机制 $\mathcal M$ 满足 $(\epsilon,\delta)$-differential privacy。

该定义对所有相邻数据集与输出采用最坏情况。较小的 $\epsilon$ 给出更紧的乘法界；$\delta$ 允许一个较小的加性失效概率，应相对于受保护单元数量与应用来选择。如果没有相邻关系、隐私单元、机制、组合发布历史与威胁模型，单独一个 $\epsilon$ 数字没有解释意义。

对于数值查询 $q$，global $L_1$ sensitivity 为

$$
\Delta_1 q
=
\max_{D\sim D'}
\|q(D)-q(D')\|_1.
$$

Laplace mechanism 发布

$$
\widetilde q(D)
=
q(D)+\operatorname{Laplace}
\left(0,\frac{\Delta_1q}{\epsilon}\right),
$$

它在相应假设下满足 pure $\epsilon$-DP。Gaussian mechanism 常用于 approximate DP，并且必须根据 sensitivity、$\epsilon$ 与 $\delta$ 校准噪声。

<details>
<summary><strong>Python：联系 sensitivity、epsilon、效用与组合发布</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(113)
true_count = 320
sensitivity = 1.0  # add/remove-one neighboring relation for a count query
draws = 50_000

for epsilon in [0.10, 0.30, 1.00, 3.00]:
    scale = sensitivity / epsilon
    released = true_count + rng.laplace(loc=0.0, scale=scale, size=draws)
    absolute_error = np.abs(released - true_count)
    print(
        f"epsilon={epsilon:>4.2f}",
        f"noise_scale={scale:>5.2f}",
        f"median_abs_error={np.median(absolute_error):>5.2f}",
        f"95th_percentile_error={np.quantile(absolute_error, 0.95):>6.2f}",
    )

# Basic sequential composition: ten epsilon=0.3 releases about the same
# protected units provide an upper bound of epsilon_total=3.0.
epsilon_per_release = 0.3
number_of_releases = 10
print(
    "basic-composition epsilon:",
    epsilon_per_release * number_of_releases,
)
```

</details>

裁剪或设界非常重要，因为无界数值可能具有无界 sensitivity。对于均值，应先把每个贡献裁剪到声明区间，再加入噪声，并核算由此引入的偏差。隐私与统计偏差是两个不同问题。

在机器学习中，**DP-SGD** 对每个 minibatch 执行：

1. 为每个样本或受保护单元计算梯度 $g_i$；
2. 把梯度裁剪到范数 $C$：

   $$
   \bar g_i
   =
   g_i\min\left(1,\frac{C}{\|g_i\|_2}\right);
   $$

3. 汇总裁剪梯度并加入 Gaussian noise；
4. 更新参数；
5. 使用 privacy accountant 组合采样步骤与发布带来的隐私损失。

裁剪限制一个单元的影响，噪声则隐藏这个已经受限的贡献。裁剪范数、noise multiplier、采样方案、步数、group privacy、checkpoint、超参数搜索与发布指标都会影响最终保证。隐私核算必须包含所有从私有数据派生并被发布的内容，不能只计算最终模型。

<div class="diagram-scroll">

![NIST 的 differential privacy pyramid 把 epsilon 放在隐私单元、算法、威胁模型、安全、访问控制与数据收集暴露之上。](assets/nist-differential-privacy-pyramid.png){fig-alt="NIST 金字塔顶部是 epsilon，中间是隐私单元和算法正确性，底部是威胁模型、安全、访问控制与数据收集暴露。"}

</div>

*图片来源与署名：[NIST Differential Privacy Pyramid](https://www.nist.gov/image/differential-privacy-pyramid)。该图强调，epsilon 主张依赖其下方的实现与运行层。*

DP 具有重要的封闭性质。如果没有使用额外私有信息，**post-processing** 不会恶化有效 DP 保证；**composition** 会让多次发布的隐私损失累积；当每个受保护的人拥有多条关联记录时，**group privacy** 会变弱。这些性质让 DP 可审计，但前提是实现与被分析机制一致。

#### **联邦学习**

联邦学习让客户端保留原始数据，同时训练共享模型。在同步 **federated averaging** 中，客户端 $k$ 从全局参数 $w_t$ 开始，执行局部优化得到 $w_{t+1}^{(k)}$，服务器再聚合：

$$
w_{t+1}
=
\sum_{k=1}^{K}
\frac{n_k}{\sum_jn_j}
w_{t+1}^{(k)}.
$$

它减少集中式数据传输、支持 data residency，并能使用分布式计算。但它本身**不能**提供机密性或 differential privacy。单个更新可能泄露信息，恶意服务器可以操纵模型状态，恶意客户端可以投毒训练，最终模型也可能记忆记录。

<div class="diagram-scroll">

![联邦学习、secure aggregation 与 differential privacy 保护不同边界。](assets/federated-learning-trust-boundaries.svg){fig-alt="客户端更新经过 secure aggregation 到达全局模型，并可再加入 differential privacy 层；图中强调三种机制处理不同威胁。"}

</div>

**Secure aggregation** 通过密码协议让服务器只学习客户端更新的聚合，而看不到每个明文更新。它在协议假设下保护更新不被服务器直接查看，却不能阻止聚合结果或最终模型泄漏。Client-level DP 可以裁剪每个客户端更新并加入校准噪声，使一个客户端是否参与的影响受到限制。客户端内部的 record-level DP 保护的是另一个隐私单元。

<details>
<summary><strong>Python：验证加权 federated averaging 与零和掩码</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(127)
client_sizes = [120, 240, 80]
clients = []
for client_id, size in enumerate(client_sizes):
    x = rng.normal(loc=0.3 * client_id, scale=1.0, size=size)
    y = 1.8 * x - 0.4 + rng.normal(scale=0.35, size=size)
    clients.append((x, y))

initial = np.array([0.0, 0.0])  # slope and intercept
learning_rate = 0.08

def gradient(parameters, x, y):
    prediction = parameters[0] * x + parameters[1]
    residual = prediction - y
    return np.array([2 * np.mean(residual * x), 2 * np.mean(residual)])

# Every client starts from the same parameters and takes one local step.
local_updates = []
for x, y in clients:
    local_updates.append(-learning_rate * gradient(initial, x, y))

weights = np.array(client_sizes) / sum(client_sizes)
federated_update = np.sum(
    np.array(local_updates) * weights[:, None], axis=0
)

all_x = np.concatenate([client[0] for client in clients])
all_y = np.concatenate([client[1] for client in clients])
central_update = -learning_rate * gradient(initial, all_x, all_y)

# Toy zero-sum masks illustrate the algebra behind secure aggregation.
# Real protocols must handle cryptography, dropouts, and malicious behavior.
mask_0 = rng.normal(size=2)
mask_1 = rng.normal(size=2)
masks = [mask_0, mask_1, -(mask_0 + mask_1)]
masked_updates = [
    client_sizes[k] * local_updates[k] + masks[k]
    for k in range(len(clients))
]
recovered_sum = np.sum(masked_updates, axis=0)
true_sum = np.sum(
    [client_sizes[k] * local_updates[k] for k in range(len(clients))],
    axis=0,
)

print("federated update:", np.round(federated_update, 6))
print("central one-step update:", np.round(central_update, 6))
print("updates match:", np.allclose(federated_update, central_update))
print("masked sum equals true sum:", np.allclose(recovered_sum, true_sum))
print("server-visible masked update 0:", np.round(masked_updates[0], 3))
```

</details>

这里之所以相等，是因为每个客户端都从相同参数开始，只执行一步 full-batch gradient，并且服务器按客户端样本数加权。在异质 non-IID 数据上进行多次局部更新后，它将不再等价于集中式 SGD，并可能产生 client drift。

联邦评估应报告：

- 按客户端与客户端类型划分的性能和校准；
- non-IID 数据与参与率变化下的收敛；
- 通信、能耗、延迟与掉线鲁棒性；
- 对 poisoning 与 backdoor 的抵抗；
- 服务器、客户端与串通威胁模型；
- secure-aggregation 假设与密钥管理；
- record-level 或 client-level 隐私核算；
- 删除、同意与模型更新政策。

**总结。**隐私是数据收集、学习、接口与发布历史共同构成的端到端属性。Membership test 诊断具体攻击；differential privacy 提供正式的分布界；secure aggregation 保护传输中的单个更新；federated learning 改变计算位置。这些机制解决不同问题，应在一个明确威胁模型下组合。


### **文档与治理**

治理把技术证据转化为可追责决策。当文档记录假设、负责人、阈值、未解决风险与变更时，它才真正有用；如果只是部署后填写的静态模板，作用非常有限。

#### **Datasheet、Model Card 与决策记录**

不同产物记录不同对象：

| 产物 | 被记录对象 | 应回答的问题 |
|---|---|---|
| Datasheet 或 data statement | 数据集 | 为什么收集？谁被代表？如何处理同意、标签、缺失、保留与访问？ |
| Model card | 拟合模型与评估 | 预期与禁止用途是什么？测试过哪些总体和环境？限制与运行阈值是什么？ |
| System card | 端到端系统 | 模型、工具、检索、人工复核、界面与 safeguard 如何交互？ |
| Experiment record | 训练运行 | 哪个代码、数据快照、特征、随机种子、指标与产物产生了该结果？ |
| Decision record | 关键决策 | 谁批准了目标、指标、阈值、缓解、例外与剩余风险，理由是什么？ |
| Monitoring plan | 已部署服务 | 哪些信号会触发警报、调查、回滚、重训练或退役？ |
| Incident report | 故障事件 | 发生了什么、谁受到影响、如何控制，以及怎样避免再次发生？ |

Model card 不能只写“AUC = 0.91”。它应明确评估总体与时间段、不确定性、子群体指标、偏移测试、校准、决策阈值、拒绝预测政策、已知盲点、隐私保证与禁止用途。链接到不可变的数据与代码版本，才能让主张可复现。

<div class="diagram-scroll">

![治理是连接范围、证据、批准、部署与监控的反馈循环。](assets/governance-evidence-loop.svg){fig-alt="生命周期从范围进入证据、决策、部署和监控，再把运行证据反馈到下一次审查。"}

</div>

治理流程应包含**阶段门槛（stage gate）**：

1. **问题门槛：**机器学习是否合适，决策目标是否合法合理？
2. **数据门槛：**来源、同意、覆盖、标签效度与受保护单元定义是否充分？
3. **评估门槛：**测试设计是否代表部署，包括关键切片与压力条件？
4. **风险门槛：**解释、鲁棒性、公平性、隐私、安全与误用发现是否位于声明容差内？
5. **部署门槛：**阈值、回退、访问控制、日志、申诉与回滚是否已经实现？
6. **监控门槛：**负责人、延迟标签、漂移测试、事件响应与退役标准是否已生效？

批准不能是一个不加区分的“responsible AI”复选框。隐私审查者、领域负责人、安全团队、受影响群体代表和模型开发者拥有不同知识与权限。决策记录应保留分歧与剩余风险，而不是把它们抹去。

监控必须沿着输入到影响的因果路径展开：

- 输入 schema 与支持；
- 预测、置信度与拒绝预测；
- 决策阈值与人工覆盖；
- 延迟、故障与访问模式；
- 延迟标签与结果质量；
- 相关切片上的错误、收益与负担；
- 申诉、投诉、事件与下游反馈；
- 隐私预算消耗与异常查询行为。

没有行动政策的漂移警报只会制造噪声。每个警报都需要负责人、时间窗口、严重程度、调查过程与允许采取的响应。重训练并不总是正确答案：concept shift 可能需要新目标或新流程，公平问题可能需要政策改变，隐私事件则可能需要先控制暴露，而不是再训练一个模型版本。

**总结。**文档应让主张能够从目的追溯到证据，再追溯到可追责决策。治理是一套运行系统，在数据、模型、用户与机构变化时持续保持这些主张有效。


### **权衡与负责任的模型选择**

不存在一个可以安全地把准确率、校准、鲁棒性、公平性、隐私、延迟、成本与人类影响压缩到一起的单一“可信度分数”。这些目标使用不同单位，也体现不同价值。因此，模型选择应采用**约束与 Pareto frontier**，而不是隐藏在 leaderboard 中的任意加权平均。

首先定义不可妥协的要求：

- 最低总体与最差群体任务性能；
- 关键分数区域中的最大 calibration error；
- 在明确压力条件下允许的最大退化；
- 运行系统真正能够处理的拒绝预测或回退容量；
- 与已记录伤害相匹配的公平容差；
- 与威胁模型相匹配的隐私与安全要求；
- 延迟、资源、可访问性与可维护性约束。

在通过门槛的模型中，应优先选择证据更清楚、复杂度更低、数据足迹更小、行为更稳定且更容易恢复的模型。很小的预测增益不应自动压过审计能力或隐私的大幅损失。

| 选择 | 可能收益 | 可能代价 | 所需证据 |
|---|---|---|---|
| 更复杂模型 | 更好的平均拟合 | 更难审计、校准与调试 | 重复样本外增益与可靠解释 |
| 激进数据增强 | 对破坏更鲁棒 | 语义扭曲或群体效应 | 现实 severity curve 与切片检查 |
| 公平约束 | 降低所选差异 | 效用变化或产生另一种差异 | Frontier、不确定性与基于伤害的论证 |
| 更强 DP | 更低参与泄漏 | 更多噪声与群体效用损失 | 隐私核算与分群体效用 |
| 更多拒绝预测 | 更低的已接受案例风险 | 服务延迟或被拒、审核人员过载 | Risk-coverage 与容量模拟 |
| 群体特定阈值 | 更好的错误率平等 | 差异化对待与运行复杂度 | 法律、伦理与结果层面的论证 |
| 联邦架构 | 更少原始数据移动 | 更多攻击面与 client drift | 系统威胁模型与客户端级评估 |

一个实际的可靠性审计可以遵循以下顺序：

1. **定义决策。**明确预测、受影响者、负责人、时间范围与每类错误的后果。
2. **指定分布与威胁。**说明部署环境、关键群体、合理破坏、攻击者、受保护单元与不支持用途。
3. **建立独立证据。**使用未接触测试数据、时间或外部验证、重复运行、置信区间与延迟结果。
4. **审计已学习行为。**结合模型特定结构与全局、局部事后方法，检验解释的保真度和稳定性。
5. **对系统施压。**扫描破坏严重程度、评估最差切片、执行自适应攻击，并测量拒绝预测下的 risk-coverage。
6. **评估伤害。**报告 base rate、混淆矩阵组成、排序曝光、校准、交叉群体、不确定性与完整决策过程。
7. **评估隐私与安全。**测试具体攻击、验证访问边界、核算正式隐私损失，并包含每一次发布。
8. **比较可行模型。**先应用硬性门槛，再检查 Pareto frontier 与运行负担。
9. **记录决策。**记录选定阈值、被拒绝替代方案、负责人、剩余风险、例外与复查日期。
10. **运行并学习。**监控结果、事件、申诉、漂移、隐私预算与回滚准备；当假设失效时让系统退役。

最终问题不是抽象地问“这个模型能解释吗？”或“这个模型公平吗？”，而是：

> 对于这个总体、环境、威胁模型与后果，现有证据是否足以支持这个由模型参与的具体决策过程，而且系统是否具备可追责的故障检测与修复方式？

有时，负责任的选择是一种受约束的可解释模型、完全由人完成的流程、随机试验、规则系统、数据收集改变，或根本不自动化。机器学习必须通过边界明确且可检验的主张赢得部署资格，而不能只依赖基准性能。
